# Integer Programming for Operations Research

This notebook provides a rigorous introduction to Integer Programming (IP) and its principal subtypes in Operations Research. It emphasizes mathematical formulation, correct variable domains, LP relaxation, exact algorithms, logical modeling, classic applications, and solution validation.

The main computational tools are `scipy.optimize.linprog` for LP relaxations and `scipy.optimize.milp` for integer and mixed-integer models. All coefficients are synthetic and intended for education and non-commercial research.

## 1. LP, Pure IP, Binary IP, and MILP

A linear optimization model can be written as

$$\min \; c^T x$$

subject to

$$Ax \le b.$$

The variable domains determine the model class:

- **LP:** all variables are continuous.
- **Pure IP:** all decision variables are integer-valued.
- **Binary IP:** all decision variables are restricted to 0 or 1.
- **MILP:** integer or binary variables coexist with continuous variables.

SciPy minimizes by default, so maximization models below minimize the negative of the objective.

In [ ]:
import math
import numpy as np
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, linprog, milp

pd.set_option('display.precision', 4)

def require_optimal(result, name='model'):
    if not result.success:
        raise RuntimeError(
            f'{name} did not solve to optimality. ' +
            f'status={result.status}, message={result.message}'
        )
    return result

## 2. Pure Integer Programming: Production Planning

Consider two products with integer production quantities. The model is

$$\max \; 7x_A + 5x_B$$

subject to

$$3x_A + 2x_B \le 17,$$

$$x_A + 2x_B \le 10,$$

$$x_A,x_B \in \mathbb{Z}_{\ge 0}.$$

In [ ]:
profit = np.array([7.0, 5.0])
pure_A = np.array([[3.0, 2.0], [1.0, 2.0]])
pure_b = np.array([17.0, 10.0])

pure_ip = require_optimal(
    milp(
        c=-profit,
        integrality=np.array([1, 1]),
        bounds=Bounds([0, 0], [np.inf, np.inf]),
        constraints=LinearConstraint(pure_A, -np.inf, pure_b),
    ),
    'pure integer production model',
)

solution = np.rint(pure_ip.x).astype(int)
pd.Series(
    {'Product A': solution[0], 'Product B': solution[1], 'Profit': -pure_ip.fun}
)

## 3. Binary Integer Programming: 0-1 Knapsack

For project selection, let $x_i=1$ when project $i$ is selected and $x_i=0$ otherwise. The classic knapsack model is

$$\max \sum_i v_i x_i$$

subject to

$$\sum_i w_i x_i \le W,$$

$$x_i\in\{0,1\}.$$

In [ ]:
projects = ['A', 'B', 'C', 'D', 'E']
value = np.array([20.0, 18.0, 14.0, 11.0, 9.0])
cost = np.array([8.0, 7.0, 6.0, 5.0, 4.0])
budget = 18.0

knapsack = require_optimal(
    milp(
        c=-value,
        integrality=np.ones(len(projects)),
        bounds=Bounds(np.zeros(len(projects)), np.ones(len(projects))),
        constraints=LinearConstraint(cost.reshape(1, -1), -np.inf, [budget]),
    ),
    '0-1 knapsack',
)

selected = np.rint(knapsack.x).astype(int)
pd.DataFrame({
    'Project': projects,
    'Selected': selected,
    'Value': value,
    'Cost': cost,
})

## 4. Mixed-Integer Linear Programming: Fixed-Charge Production

Let $y\in\{0,1\}$ indicate whether a production line is activated and let $x\ge0$ be a continuous production quantity.

$$\min \; 40y + 3x$$

subject to

$$x \ge 12,$$

$$x \le 20y,$$

$$y\in\{0,1\}, \quad x\ge0.$$

This is a genuine MILP because $x$ is continuous and $y$ is binary.

In [ ]:
# Variable order: [x, y]
c = np.array([3.0, 40.0])
A = np.array([[1.0, 0.0], [1.0, -20.0]])
lb = np.array([12.0, -np.inf])
ub = np.array([np.inf, 0.0])

fixed_charge = require_optimal(
    milp(
        c=c,
        integrality=np.array([0, 1]),
        bounds=Bounds([0.0, 0.0], [np.inf, 1.0]),
        constraints=LinearConstraint(A, lb, ub),
    ),
    'fixed-charge MILP',
)

pd.Series({
    'Production x': fixed_charge.x[0],
    'Open y': fixed_charge.x[1],
    'Total cost': fixed_charge.fun,
})

## 5. LP Relaxation and the Integrality Gap

The LP relaxation removes integer restrictions. For a maximization problem, its optimum is an upper bound on the integer optimum. Consider

$$\max \; 5x_1+4x_2$$

subject to

$$6x_1+4x_2\le24,$$

$$x_1+2x_2\le6,$$

$$x_1,x_2\ge0.$$

The LP optimum is fractional at $(3,1.5)$ with objective 21, while the integer optimum is 20.

In [ ]:
objective = np.array([5.0, 4.0])
A_gap = np.array([[6.0, 4.0], [1.0, 2.0]])
b_gap = np.array([24.0, 6.0])

lp_relaxation = require_optimal(
    linprog(
        c=-objective, A_ub=A_gap, b_ub=b_gap,
        bounds=[(0, None), (0, None)], method='highs'
    ),
    'LP relaxation',
)

integer_model = require_optimal(
    milp(
        c=-objective,
        integrality=np.array([1, 1]),
        bounds=Bounds([0, 0], [np.inf, np.inf]),
        constraints=LinearConstraint(A_gap, -np.inf, b_gap),
    ),
    'integer counterpart',
)

pd.DataFrame({
    'Model': ['LP relaxation', 'Integer model'],
    'x1': [lp_relaxation.x[0], integer_model.x[0]],
    'x2': [lp_relaxation.x[1], integer_model.x[1]],
    'Objective': [-lp_relaxation.fun, -integer_model.fun],
})

## 6. Branch-and-Bound from First Principles

Branch-and-Bound repeatedly solves LP relaxations. A node is pruned when it is infeasible, when its LP solution is already integer, or when its bound cannot improve the incumbent. The following implementation is intentionally small and pedagogical.

In [ ]:
def branch_and_bound_max(c, A_ub, b_ub, lower=None, upper=None, tol=1e-9):
    c = np.asarray(c, dtype=float)
    n = len(c)
    lower = np.zeros(n) if lower is None else np.asarray(lower, dtype=float)
    upper = np.full(n, np.inf) if upper is None else np.asarray(upper, dtype=float)

    incumbent_value = -np.inf
    incumbent_x = None
    nodes = []
    stack = [(lower.copy(), upper.copy(), 0)]

    while stack:
        node_lb, node_ub, depth = stack.pop()
        bounds = [
            (float(node_lb[i]), None if np.isinf(node_ub[i]) else float(node_ub[i]))
            for i in range(n)
        ]
        lp = linprog(c=-c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

        if not lp.success:
            nodes.append((depth, 'infeasible', None, None))
            continue

        lp_value = -lp.fun
        if lp_value <= incumbent_value + tol:
            nodes.append((depth, 'bound-pruned', lp.x.copy(), lp_value))
            continue

        fractional = [
            i for i, value in enumerate(lp.x)
            if abs(value - round(value)) > tol
        ]

        if not fractional:
            incumbent_value = lp_value
            incumbent_x = np.rint(lp.x).astype(int)
            nodes.append((depth, 'integer-incumbent', lp.x.copy(), lp_value))
            continue

        j = fractional[0]
        value = lp.x[j]
        nodes.append((depth, f'branch on x{j+1}={value:.4f}', lp.x.copy(), lp_value))

        right_lb = node_lb.copy()
        right_lb[j] = max(right_lb[j], math.ceil(value))
        if right_lb[j] <= node_ub[j]:
            stack.append((right_lb, node_ub.copy(), depth + 1))

        left_ub = node_ub.copy()
        left_ub[j] = min(left_ub[j], math.floor(value))
        if node_lb[j] <= left_ub[j]:
            stack.append((node_lb.copy(), left_ub, depth + 1))

    return incumbent_x, incumbent_value, nodes

bb_x, bb_value, bb_nodes = branch_and_bound_max(objective, A_gap, b_gap)

assert np.allclose(bb_x, np.rint(integer_model.x).astype(int))
assert math.isclose(bb_value, -integer_model.fun, abs_tol=1e-9)

pd.DataFrame([
    {
        'Depth': d,
        'Action': a,
        'LP solution': None if x is None else np.round(x, 4).tolist(),
        'LP bound': v,
    }
    for d, a, x, v in bb_nodes
])

## 7. Cutting Planes and Valid Inequalities

A valid cut removes fractional LP solutions without removing feasible integer solutions. For the binary knapsack constraint

$$6x_1+4x_2+3x_3\le8,$$

items 1 and 2 cannot both be selected because $6+4>8$. Therefore

$$x_1+x_2\le1$$

is a valid cover inequality. Modern solvers combine branching and many classes of cuts in Branch-and-Cut.

In [ ]:
cover_value = np.array([10.0, 9.0, 6.0])
cover_weight = np.array([6.0, 4.0, 3.0])
cover_capacity = 8.0

lp_without_cut = require_optimal(
    linprog(
        c=-cover_value,
        A_ub=cover_weight.reshape(1, -1),
        b_ub=[cover_capacity],
        bounds=[(0, 1)] * 3,
        method='highs',
    ),
    'knapsack LP without cut',
)

A_with_cut = np.vstack([cover_weight, [1.0, 1.0, 0.0]])
b_with_cut = np.array([cover_capacity, 1.0])

lp_with_cut = require_optimal(
    linprog(
        c=-cover_value,
        A_ub=A_with_cut,
        b_ub=b_with_cut,
        bounds=[(0, 1)] * 3,
        method='highs',
    ),
    'knapsack LP with cover cut',
)

pd.DataFrame({
    'Model': ['LP without cover cut', 'LP with cover cut'],
    'x1': [lp_without_cut.x[0], lp_with_cut.x[0]],
    'x2': [lp_without_cut.x[1], lp_with_cut.x[1]],
    'x3': [lp_without_cut.x[2], lp_with_cut.x[2]],
    'Objective': [-lp_without_cut.fun, -lp_with_cut.fun],
})

## 8. Logical Constraints and Big-M

A standard linking constraint is

$$x\le My,$$

where $y\in\{0,1\}$. If $y=0$, a nonnegative $x$ is forced to zero. If $y=1$, $x$ can be positive up to $M$. The strongest formulation uses the smallest valid $M$, preferably a real capacity or physical bound rather than an arbitrary huge number.

In [ ]:
# Variable order: [ship_1, ship_2, open_1, open_2]
shipping_cost = np.array([2.0, 3.0])
fixed_cost = np.array([25.0, 18.0])
capacity = np.array([10.0, 9.0])
demand = 12.0

c = np.r_[shipping_cost, fixed_cost]
A = np.array([
    [-1.0, -1.0, 0.0, 0.0],
    [1.0, 0.0, -capacity[0], 0.0],
    [0.0, 1.0, 0.0, -capacity[1]],
])
ub = np.array([-demand, 0.0, 0.0])

facility_activation = require_optimal(
    milp(
        c=c,
        integrality=np.array([0, 0, 1, 1]),
        bounds=Bounds([0, 0, 0, 0], [np.inf, np.inf, 1, 1]),
        constraints=LinearConstraint(A, -np.inf, ub),
    ),
    'facility activation MILP',
)

pd.Series(
    facility_activation.x,
    index=['Ship F1', 'Ship F2', 'Open F1', 'Open F2'],
)

## 9. Facility Location

The uncapacitated facility-location model uses binary facility variables $y_i$ and binary assignment variables $x_{ij}$.

$$\min \sum_i f_i y_i + \sum_i\sum_j c_{ij}x_{ij}$$

subject to

$$\sum_i x_{ij}=1 \quad \forall j,$$

$$x_{ij}\le y_i \quad \forall i,j.$$

In [ ]:
facilities = ['F1', 'F2', 'F3']
customers = ['C1', 'C2', 'C3', 'C4']
fixed = np.array([14.0, 16.0, 13.0])
assignment_cost = np.array([
    [4.0, 6.0, 9.0, 8.0],
    [5.0, 4.0, 7.0, 6.0],
    [8.0, 7.0, 3.0, 4.0],
])

n_f, n_c = len(facilities), len(customers)
def x_index(i, j):
    return n_f + i * n_c + j

n_vars = n_f + n_f * n_c
c = np.zeros(n_vars)
c[:n_f] = fixed
for i in range(n_f):
    for j in range(n_c):
        c[x_index(i, j)] = assignment_cost[i, j]

rows, lower, upper = [], [], []
for j in range(n_c):
    row = np.zeros(n_vars)
    for i in range(n_f):
        row[x_index(i, j)] = 1.0
    rows.append(row); lower.append(1.0); upper.append(1.0)

for i in range(n_f):
    for j in range(n_c):
        row = np.zeros(n_vars)
        row[x_index(i, j)] = 1.0
        row[i] = -1.0
        rows.append(row); lower.append(-np.inf); upper.append(0.0)

facility_location = require_optimal(
    milp(
        c=c,
        integrality=np.ones(n_vars),
        bounds=Bounds(np.zeros(n_vars), np.ones(n_vars)),
        constraints=LinearConstraint(np.vstack(rows), np.array(lower), np.array(upper)),
    ),
    'facility location',
)

open_decisions = np.rint(facility_location.x[:n_f]).astype(int)
assignments = np.rint(facility_location.x[n_f:]).astype(int).reshape(n_f, n_c)
print('Total cost:', facility_location.fun)
print(pd.Series(open_decisions, index=facilities, name='Open'))
pd.DataFrame(assignments, index=facilities, columns=customers)

## 10. Assignment and Parallel-Machine Scheduling

Let $x_{jm}=1$ if job $j$ is assigned to machine $m$ and let $C_{max}$ be the continuous makespan variable.

$$\min C_{max}$$

subject to

$$\sum_m x_{jm}=1 \quad \forall j,$$

$$\sum_j p_jx_{jm}\le C_{max} \quad \forall m.$$

This is a MILP because assignment variables are binary and makespan is continuous.

In [ ]:
jobs = ['J1', 'J2', 'J3', 'J4', 'J5']
machines = ['M1', 'M2']
processing = np.array([7.0, 6.0, 5.0, 4.0, 3.0])
n_j, n_m = len(jobs), len(machines)
n_x = n_j * n_m
makespan_idx = n_x
n_vars = n_x + 1

def assign_index(j, m):
    return j * n_m + m

c = np.zeros(n_vars)
c[makespan_idx] = 1.0
rows, lower, upper = [], [], []

for j in range(n_j):
    row = np.zeros(n_vars)
    for m in range(n_m):
        row[assign_index(j, m)] = 1.0
    rows.append(row); lower.append(1.0); upper.append(1.0)

for m in range(n_m):
    row = np.zeros(n_vars)
    for j in range(n_j):
        row[assign_index(j, m)] = processing[j]
    row[makespan_idx] = -1.0
    rows.append(row); lower.append(-np.inf); upper.append(0.0)

scheduling = require_optimal(
    milp(
        c=c,
        integrality=np.r_[np.ones(n_x), 0],
        bounds=Bounds(np.zeros(n_vars), np.r_[np.ones(n_x), np.inf]),
        constraints=LinearConstraint(np.vstack(rows), np.array(lower), np.array(upper)),
    ),
    'parallel machine scheduling',
)

assignment = np.rint(scheduling.x[:n_x]).astype(int).reshape(n_j, n_m)
assignment_df = pd.DataFrame(assignment, index=jobs, columns=machines)
assignment_df, scheduling.x[makespan_idx]

## 11. Network Design

A common fixed-charge network formulation uses continuous flow $f_{ij}$ and a binary arc-activation variable $y_{ij}$ with

$$0\le f_{ij}\le u_{ij}y_{ij}.$$

The linking constraint ensures that an arc can carry flow only when it is activated.

In [ ]:
arcs = [('S', 'A'), ('S', 'B'), ('A', 'T'), ('B', 'T'), ('A', 'B')]
capacity = np.array([8.0, 7.0, 8.0, 7.0, 3.0])
variable_cost = np.array([1.0, 1.2, 1.0, 0.9, 0.4])
fixed_cost = np.array([5.0, 4.0, 5.0, 4.0, 2.0])
required_flow = 10.0
n_a = len(arcs)

c = np.r_[variable_cost, fixed_cost]
integrality = np.r_[np.zeros(n_a), np.ones(n_a)]
lb_vars = np.zeros(2 * n_a)
ub_vars = np.r_[capacity, np.ones(n_a)]

nodes = ['S', 'A', 'B', 'T']
balance = {'S': required_flow, 'A': 0.0, 'B': 0.0, 'T': -required_flow}
rows, lower, upper = [], [], []

for node in nodes:
    row = np.zeros(2 * n_a)
    for a, (u, v) in enumerate(arcs):
        if u == node:
            row[a] += 1.0
        if v == node:
            row[a] -= 1.0
    rows.append(row); lower.append(balance[node]); upper.append(balance[node])

for a in range(n_a):
    row = np.zeros(2 * n_a)
    row[a] = 1.0
    row[n_a + a] = -capacity[a]
    rows.append(row); lower.append(-np.inf); upper.append(0.0)

network = require_optimal(
    milp(
        c=c,
        integrality=integrality,
        bounds=Bounds(lb_vars, ub_vars),
        constraints=LinearConstraint(np.vstack(rows), np.array(lower), np.array(upper)),
    ),
    'network design',
)

pd.DataFrame({
    'Arc': [f'{u}->{v}' for u, v in arcs],
    'Flow': network.x[:n_a],
    'Activated': np.rint(network.x[n_a:]).astype(int),
    'Capacity': capacity,
})

## 12. Multi-Objective Integer Programming

Common approaches include weighted sums, epsilon-constraint formulations, and lexicographic optimization. The weighted-sum example below trades financial return against risk exposure.

In [ ]:
returns = np.array([12.0, 10.0, 9.0, 7.0])
risk = np.array([8.0, 5.0, 4.0, 2.0])
capital = np.array([6.0, 5.0, 4.0, 3.0])
budget = 11.0

def solve_weighted_project_selection(risk_weight):
    score = returns - risk_weight * risk
    result = require_optimal(
        milp(
            c=-score,
            integrality=np.ones(4),
            bounds=Bounds(np.zeros(4), np.ones(4)),
            constraints=LinearConstraint(capital.reshape(1, -1), -np.inf, [budget]),
        ),
        f'weighted project selection (risk_weight={risk_weight})',
    )
    x = np.rint(result.x).astype(int)
    return {
        'Risk weight': risk_weight,
        'Selected': x.tolist(),
        'Return': float(returns @ x),
        'Risk': float(risk @ x),
        'Capital': float(capital @ x),
    }

pd.DataFrame(solve_weighted_project_selection(w) for w in [0.0, 0.25, 0.5, 1.0])

## 13. Heuristics, Complexity, and Formulation Quality

Exact methods seek both a feasible solution and a proof of optimality. Modern MILP solvers combine LP relaxation, Branch-and-Bound, cutting planes, presolve, primal heuristics, and sophisticated branching rules.

Heuristics such as greedy methods, local search, tabu search, simulated annealing, genetic algorithms, and large neighborhood search can find good feasible solutions quickly, but they do not generally prove global optimality.

Important computational issues include weak LP relaxations, excessively large Big-M constants, symmetry, poor variable bounds, and unnecessary integer restrictions. Two mathematically equivalent formulations can have very different computational performance.

## 14. Solution Validation and Common Modeling Mistakes

Always check solver status before interpreting decision variables. Recompute the objective independently, verify constraint feasibility, test integrality within a numerical tolerance, and check application-specific invariants.

Common mistakes include:

1. Solving an LP and simply rounding the solution.
2. Declaring variables integer when discreteness is not required.
3. Using an arbitrary huge Big-M instead of a tight valid bound.
4. Confusing binary and general integer variables.
5. Treating a feasible incumbent as a proven optimum.
6. Ignoring LP-relaxation quality.
7. Writing logical implications without testing every binary state.

In [ ]:
# Independent validation of the pure IP example.
x = np.rint(pure_ip.x).astype(int)
assert np.all(x >= 0)
assert np.allclose(x, pure_ip.x, atol=1e-7)
assert np.all(pure_A @ x <= pure_b + 1e-9)
assert math.isclose(profit @ x, -pure_ip.fun, abs_tol=1e-9)
print('Pure IP solution passed independent validation.')

## 15. Summary and Further Study

Integer Programming is the standard mathematical framework for discrete optimization. Pure IP uses integer decision variables, Binary IP restricts decisions to 0 or 1, and MILP combines discrete and continuous decisions. Core concepts include LP relaxation, integrality gaps, Branch-and-Bound, cutting planes, Branch-and-Cut, Big-M linking, and solution validation.

Recommended next topics include Gomory cuts, lifted cover inequalities, set covering and set partitioning, Benders decomposition, column generation, branch-and-price, vehicle routing formulations, stochastic integer programming, robust optimization, and advanced multi-objective MILP.